# Uçuş Veri Seti Kapsamlı Analizi
Bu notebook, uçuş verileri üzerinde temel istatistikler, aykırı değer tespiti (IQR), korelasyon analizi ve K-Means kümeleme çalışmalarını içermektedir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# Estetik ayarlar
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

# Veriyi yükleme
df = pd.read_csv('Clean_Dataset.csv', index_col=0)
df.head()

## 1. Temel İstatistiksel Analiz (Describe)
Veri setindeki sayısal değişkenlerin merkezi eğilim ve yayılım ölçülerini inceliyoruz.

In [ ]:
df.describe()

## 2. Detaylı Aykırı Değer Tespiti (IQR Metodu)
Her değişken için Q1, Q3 ve IQR değerlerini hesaplayarak alt ve üst sınırları belirliyoruz.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
    print(f"Sütun: {col}")
    print(f"  IQR: {IQR:.2f} | Sınırlar: [{lower_bound:.2f}, {upper_bound:.2f}]")
    print(f"  Aykırı Değer Sayısı: {len(outliers)}\n" + "-"*30)

## 3. Korelasyon Analizi
Değişkenler arasındaki doğrusal ilişkiyi inceliyoruz.

In [ ]:
corr_matrix = df[numeric_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.3f')
plt.title('Korelasyon Matrisi Heatmap')
plt.show()

## 4. Gelişmiş Kümeleme (K-Means)
Fiyat ve Süre değişkenlerine göre veriyi 4 ana segmente ayırıyoruz.

In [ ]:
X = df[['price', 'duration']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

sns.scatterplot(data=df.sample(5000), x='duration', y='price', hue='cluster', palette='viridis')
plt.title('K-Means Kümeleme: Fiyat vs Süre')
plt.show()

## 5. Görselleştirmeler
Değişkenlerin dağılımlarını görsel olarak analiz ediyoruz.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, col in enumerate(numeric_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='skyblue')
    axes[i].set_title(f'{col} Dağılımı')
plt.tight_layout()
plt.show()